In [1]:
import pandas as pd
import numpy as np

In [2]:
df_columns=['rownum', 'A1', 'A2', 'A3', 'B', 'C1', 'C2', 'C3', 'xA', 'yA',
       'zA', 'xC', 'yC', 'zC', 'Eg', 'Eg-type', 'rA1', 'rA2', 'rA3', 'rA', 'rB', 'rC1 ', 'rC2',
       'rC3', 'rC', 'TF', 'OF', "duplicated"]
c_columns_drop=["Unnamed: 0", "link"]
data_path="../data/"

perovskites_dataset_url = data_path+"dataset(Sona).xlsx"
constraints_dataset_url = data_path+"constants.xlsx"

In [3]:
df = pd.read_excel(perovskites_dataset_url, skiprows=1)[df_columns]
##removing some whitespace from some column names
df.columns = [col.strip() for col in df.columns]
c_df = pd.read_excel(constraints_dataset_url).drop(columns=c_columns_drop).set_index("name")

In [4]:
columns_with_nulls = df.columns[df.isnull().any()]

null_values_per_column = {}

for col in columns_with_nulls:
    null_values_per_column[col] = df[df[col].isnull()]  # Filter rows with nulls for this column

In [5]:
c_df.head()

,Pb,Cs,MA,FA,Cl,Br,I
name,,,,,,,
r,1.19,1.67,2.16,2.53,1.81,1.96,2.2
pn,6.00,4.00,NaN,NaN,3.00,4.00,5
gn,14.00,6.00,NaN,NaN,17.00,17.00,17
EA,35.10,45.50,NaN,NaN,227.00,244.00,243.5
IE,7.42,3.89,NaN,NaN,12.97,11.81,10.45


In [6]:
df.columns

Index(['rownum', 'A1', 'A2', 'A3', 'B', 'C1', 'C2', 'C3', 'xA', 'yA', 'zA',
       'xC', 'yC', 'zC', 'Eg', 'Eg-type', 'rA1', 'rA2', 'rA3', 'rA', 'rB',
       'rC1', 'rC2', 'rC3', 'rC', 'TF', 'OF', 'duplicated'],
      dtype='object')

In [7]:
###preparing input columns of x,y,z
for col in ['xA', 'yA', 'zA', 'xC', 'yC', 'zC']:
    df[col]=df[col].fillna(0)

In [8]:
###inserting element radii
def calculate_r_value(name):
  value = 0
  if name in c_df.columns:
    print("'"+name+"'")
    value = c_df.loc['r', name]
  return value

for col in ['rA1', 'rA2', 'rA3', 'rB', 'rC1', 'rC2', 'rC3']:
  df[col] = calculate_r_value(df.loc[0, col[1:]])

'Cs'
'MA'
'FA'
'Pb'
'Cl'
'Br'
'I'


In [9]:
###calculating radii of A and C
df["rA"] = df["xA"]*df["rA1"]+df["yA"]*df["rA2"]+df["zA"]*df["rA3"]
df["rC"] = df["xC"]*df["rC1"]+df["yC"]*df["rC2"]+df["zC"]*df["rC3"]

In [10]:
###calculating the composite columns
df["TF"]=(df["rA"]+df["rC"])/np.sqrt(2)/(df["rB"]+df["rC"])
df["OF"]=df["rB"]/df["rC"]

In [11]:
df.head()

,rownum,A1,A2,A3,B,C1,C2,C3,xA,yA,...,rA3,rA,rB,rC1,rC2,rC3,rC,TF,OF,duplicated
0,1.0,Cs,MA,FA,Pb,Cl,Br,I,1.0,0.0,...,2.53,1.67,1.19,1.81,1.96,2.2,2.2000,0.807228,0.540909,yes
1,2.0,Cs,MA,FA,Pb,Cl,Br,I,1.0,0.0,...,2.53,1.67,1.19,1.81,1.96,2.2,2.1208,0.809623,0.561109,NaN
2,3.0,Cs,MA,FA,Pb,Cl,Br,I,1.0,0.0,...,2.53,1.67,1.19,1.81,1.96,2.2,2.0392,0.812214,0.583562,NaN
3,4.0,Cs,MA,FA,Pb,Cl,Br,I,1.0,0.0,...,2.53,1.67,1.19,1.81,1.96,2.2,1.9600,0.814856,0.607143,NaN
4,5.0,Cs,MA,FA,Pb,Cl,Br,I,1.0,0.0,...,2.53,1.67,1.19,1.81,1.96,2.2,2.0713,0.811179,0.574518,NaN


In [12]:
df["duplicated"] = (df["duplicated"]=="yes").astype(int)
df = df[df["Eg-type"]!="calculated"]

In [13]:
df.to_csv("../data/prepared_data_Sona.csv")